# XGBoost — Flight Delay Cause Prediction

Dự đoán 5 nguyên nhân delay theo multi-label classification. Giữ cả chuyến có và không có nguyên nhân; không lọc arr_delay >= 15.


## Cài đặt thư viện

```bash
pip install xgboost scikit-learn pandas matplotlib
```


In [ ]:

import time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

DATA_PATH = Path("../data/processed/split")

TARGET_COLUMNS = [
    "carrier_delay", "weather_delay", "nas_delay",
    "security_delay", "late_aircraft_delay"
]

CATEGORICAL_FEATURES = ["op_unique_carrier", "origin", "dest"]

NUMERICAL_FEATURES = [
    "month", "day_of_month", "day_of_week",
    "op_carrier_fl_num", "crs_dep_time", "crs_arr_time",
    "crs_elapsed_time", "distance"
]

FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

train_df = pd.read_csv(DATA_PATH / "train.csv")
validation_df = pd.read_csv(DATA_PATH / "validation.csv")
test_df = pd.read_csv(DATA_PATH / "test.csv")

X_train = train_df[FEATURE_COLUMNS].copy()
X_test = test_df[FEATURE_COLUMNS].copy()

Y_train = train_df[TARGET_COLUMNS].astype(int)
Y_test = test_df[TARGET_COLUMNS].astype(int)

print("Train:", X_train.shape)
print("Validation:", validation_df.shape)
print("Test:", X_test.shape)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

OUTPUT_PATH = Path("../reports/xgboost")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
(OUTPUT_PATH / "confusion_matrices").mkdir(parents=True, exist_ok=True)

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ("num", "passthrough", NUMERICAL_FEATURES)
])

start = time.perf_counter()
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)
preprocessing_time = time.perf_counter() - start

results = []

for target in TARGET_COLUMNS:
    model = XGBClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss"
    )

    start = time.perf_counter()
    model.fit(X_train_encoded, Y_train[target])
    training_time = time.perf_counter() - start

    y_pred = model.predict(X_test_encoded)

    results.append({
        "target": target,
        "training_time_seconds": training_time,
        "accuracy": accuracy_score(Y_test[target], y_pred),
        "precision": precision_score(Y_test[target], y_pred, zero_division=0),
        "recall": recall_score(Y_test[target], y_pred, zero_division=0),
        "f1": f1_score(Y_test[target], y_pred, zero_division=0)
    })

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(
        confusion_matrix(Y_test[target], y_pred),
        display_labels=["No", "Yes"]
    ).plot(ax=ax)
    ax.set_title(f"XGBoost - {target}")
    fig.tight_layout()
    fig.savefig(
        OUTPUT_PATH / "confusion_matrices" / f"{target}.png",
        dpi=150
    )
    plt.close(fig)

metrics_df = pd.DataFrame(results)
metrics_df.to_csv(OUTPUT_PATH / "metrics.csv", index=False)

pd.DataFrame([{
    "preprocessing_time_seconds": preprocessing_time
}]).to_csv(
    OUTPUT_PATH / "preprocessing_time.csv",
    index=False
)

metrics_df
